# Focus Fatigue — Full Pipeline

Thin runner that calls `src.pipeline` — no inline pipeline logic.

## Setup
1. Set `DATA_ROOT` to your data directory
2. Optionally set `MATCH_IDS` to limit which matches to process
3. Run all cells

In [ ]:
import sys
from pathlib import Path

# ── POINT THIS TO YOUR DATA DIRECTORY ──
DATA_ROOT = "/mnt/usb/conor_downloads/team_mappings"

# Leave empty to auto-discover, or specify exact match IDs
MATCH_IDS = []  # e.g. ["2215790"] or [] for all

# Optional: limit frames per match for quick testing
TEST_FRAMES = None  # e.g. 5000

# ── Resolved paths ──
TRACKING_DIR = Path(DATA_ROOT) / "tracking"
SAMPLE_DIR = Path(DATA_ROOT) / "sample"

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Data root:    {DATA_ROOT}")
print(f"Tracking:     {TRACKING_DIR}")
print(f"Sample:       {SAMPLE_DIR}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ── Configure paths ──
from src.pressure.config import DEFAULT_CONFIG
from src.signals.config import DEFAULT_SIGNAL_CONFIG

# Override paths in config
DEFAULT_CONFIG.tracking_dir = str(TRACKING_DIR)
DEFAULT_CONFIG.sample_dir = str(SAMPLE_DIR)
DEFAULT_CONFIG.output_dir = str(PROJECT_ROOT / "outputs" / "pressure_exposure")
DEFAULT_SIGNAL_CONFIG.output_root = str(PROJECT_ROOT / "outputs" / "signals")

# Also set team mappings path used by drift signal
import src.signals.drift as drift_module
team_map_path = Path(DATA_ROOT) / "team_mappings" / "team_mappings.csv"
if not team_map_path.exists():
    team_map_path = Path(DATA_ROOT) / "team_mappings.csv"
drift_module._TEAM_MAPPINGS_PATH = team_map_path

import os
os.makedirs(DEFAULT_CONFIG.output_dir, exist_ok=True)
os.makedirs(DEFAULT_SIGNAL_CONFIG.output_root, exist_ok=True)
for sig in ["positional_drift", "shift_latency", "pressing_accuracy", "transition_latency"]:
    os.makedirs(os.path.join(DEFAULT_SIGNAL_CONFIG.output_root, sig), exist_ok=True)

print(f"Pressure output: {DEFAULT_CONFIG.output_dir}")
print(f"Signals output:  {DEFAULT_SIGNAL_CONFIG.output_root}")
print(f"Team map:        {team_map_path} (exists: {team_map_path.exists()})")

In [ ]:
# ── Discover matches ──
if MATCH_IDS:
    matches = MATCH_IDS
else:
    matches = sorted([d.name for d in TRACKING_DIR.iterdir()
                      if d.is_dir() and (d / "tracking.parquet").exists()])
    if not matches:
        matches = sorted([d.name for d in SAMPLE_DIR.iterdir()
                          if d.is_dir() and (d / "tracking.parquet").exists()])

if not matches:
    print("❌ No matches found!")
else:
    print(f"Found {len(matches)} matches to process:")
    for m in matches[:5]:
        print(f"  • {m}")
    if len(matches) > 5:
        print(f"  ... and {len(matches)-5} more")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 1: Model 1 — Pressure Exposure
# ═══════════════════════════════════════════════════════════════
# Checkpoint: each match is saved as CSV immediately.
# After all matches, this cell also saves the summary dict to a JSON
# checkpoint so a crash after Phase 1 does not lose the metadata.

import json
from src.pipeline import run_model1_on_match

CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PHASE1_CHECKPOINT_PATH = CHECKPOINT_DIR / "phase1_summary.json"

print("=" * 70)
print("PHASE 1: MODEL 1 — PRESSURE EXPOSURE")
print("=" * 70)

# Try to resume from checkpoint
model1_results = {}
resumed = 0
if PHASE1_CHECKPOINT_PATH.exists():
    try:
        with open(PHASE1_CHECKPOINT_PATH) as f:
            model1_results = json.load(f)
        resumed = len(model1_results)
        print(f"  ℹ️  Resumed from checkpoint: {resumed} matches already completed")
    except (json.JSONDecodeError, KeyError):
        model1_results = {}

phase1_start_ct = len(model1_results)
for idx, match_id in enumerate(matches):
    if match_id in model1_results:
        continue  # already done
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        print(f"  ⚠️  [{idx+1}/{len(matches)}] {match_id}: not found, skipping")
        continue
    result = run_model1_on_match(match_id, match_path, DEFAULT_CONFIG, nrows=TEST_FRAMES)
    model1_results[match_id] = result
    # Save checkpoint after each match (append mode)
    with open(PHASE1_CHECKPOINT_PATH, "w") as f:
        json.dump(model1_results, f, indent=2, default=str)

print(f"\n{'=' * 70}")
print(f"Model 1 complete: {len(model1_results)} matches")
print(f"{'=' * 70}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 2: All Signals (with per-match timing)
# ═══════════════════════════════════════════════════════════════

import time
from src.pipeline import run_signals_on_match, list_signal_descriptions
from src.signals.registry import list_signals

print("=" * 70)
print("PHASE 2: ALL SIGNALS")
print("=" * 70)

registered = list_signals()
print(f"Registered signals ({len(registered)}): {registered}\n")

signal_results = {}
for idx, match_id in enumerate(matches):
    match_path = TRACKING_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        match_path = SAMPLE_DIR / match_id / "tracking.parquet"
    if not match_path.exists():
        continue
    t_match = time.time()
    result = run_signals_on_match(match_id, match_path, DEFAULT_CONFIG,
                                   DEFAULT_SIGNAL_CONFIG,
                                   str(TRACKING_DIR), nrows=TEST_FRAMES)
    match_elapsed = time.time() - t_match
    signal_results[match_id] = result
    
    # Per-match timing breakdown
    sig_times = result["signals"]
    if sig_times:
        timing = []
        for sn in registered:
            info = sig_times.get(sn, {})
            et = info.get("elapsed_s", 0)
            rows = info.get("rows", 0)
            status = "❌" if "error" in info else "✅"
            timing.append(f"{status} {sn}: {rows}r/{et:.1f}s")
        print(f"  [{idx+1}/{len(matches)}] {match_id}: {match_elapsed:.0f}s total — "
              + ", ".join(timing))

# Save Phase 2 checkpoint as well
PHASE2_CHECKPOINT_PATH = CHECKPOINT_DIR / "phase2_summary.json"
with open(PHASE2_CHECKPOINT_PATH, "w") as f:
    json.dump(signal_results, f, indent=2, default=str)

print(f"\n{'=' * 70}")
print(f"Signals complete: {len(signal_results)} matches")
print(f"{'=' * 70}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PHASE 3: Merge into Unified Dataset
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("PHASE 3: MERGE OUTPUTS")
print("=" * 70)

from src.merge_outputs import merge_all

unified = merge_all(
    signals_dir=str(PROJECT_ROOT / "outputs" / "signals"),
    pressure_dir=str(PROJECT_ROOT / "outputs" / "pressure_exposure"),
    output_path=str(PROJECT_ROOT / "outputs" / "unified_fatigue_dataset.parquet"),
)
print(f"Unified dataset: {len(unified)} rows × {len(unified.columns)} columns")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════

print("=" * 70)
print("PIPELINE COMPLETE — SUMMARY")
print("=" * 70)
print(f"  Matches attempted: {len(matches)}")
print(f"  Model 1 success:   {len(model1_results)}")
print(f"  Signals success:   {len(signal_results)}")
print()

if model1_results:
    total_players = sum(r.get("n_players", 0) for r in model1_results.values() if "error" not in r)
    high = sum(r.get("high", 0) for r in model1_results.values() if "error" not in r)
    low = sum(r.get("low", 0) for r in model1_results.values() if "error" not in r)
    p1_time = sum(r.get("elapsed_s", 0) for r in model1_results.values() if "error" not in r)
    print(f"  Model 1:")
    print(f"    Players:      {total_players}")
    print(f"    High blocks:  {high}")
    print(f"    Low blocks:   {low}")
    print(f"    Total time:   {p1_time:.0f}s ({p1_time/60:.1f}m)")
    print()

if signal_results:
    print(f"  {'Signal':<25} {'Rows':>8} {'Errors':>8} {'Time':>8}")
    print(f"  {'─' * 54}")
    for sig_name in registered:
        total_rows = sum(r["signals"].get(sig_name, {}).get("rows", 0) for r in signal_results.values())
        total_time = sum(r["signals"].get(sig_name, {}).get("elapsed_s", 0) for r in signal_results.values())
        errors = [mid for mid, r in signal_results.items() if r["signals"].get(sig_name, {}).get("error")]
        err_str = str(len(errors)) if errors else "─"
        time_str = f"{total_time:.0f}s" if total_time else "─"
        print(f"  {sig_name:<25} {total_rows:>8,} {err_str:>8} {time_str:>8}")

print()
print(f"  Output files:")
print(f"    Pressure:  {DEFAULT_CONFIG.output_dir}/")
print(f"    Signals:   {DEFAULT_SIGNAL_CONFIG.output_root}/{{signal}}/{{match}}.csv")
print(f"    Unified:   {PROJECT_ROOT / 'outputs' / 'unified_fatigue_dataset.parquet'}")
print(f"    Checkpoints: {CHECKPOINT_DIR}/")